In [ ]:
import sys
from contextlib import redirect_stdout 
import io
import pandas as pd

from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve()))

df = pd.read_parquet('../ssi_all_features.parquet')
df.shape

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted.shape

In [ ]:
from ssi.preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted, defer_imputation=True)

In [ ]:
df_pivoted_preprocessed.shape

In [ ]:
from ssi.feature_filter import feature_filter

excluded = {'birthid', 'mother_patid', 'target_variable'}
X = df_pivoted_preprocessed.drop(columns=list(excluded))
y = df_pivoted_preprocessed['target_variable']
mother_patid = df_pivoted_preprocessed['mother_patid']
with redirect_stdout(io.StringIO()):
    df_pivoted_filtered = feature_filter(X.copy(), skip_medadmin_ingredient=True)

In [ ]:
df_pivoted_filtered.shape

In [ ]:
# Use the same full-model predictor pool as the main timing notebook.
or_timings_features = [x for x in df_pivoted_filtered.columns if x.startswith('or_timings')]
all_features = [
    x for x in df_pivoted_filtered.columns if x not in or_timings_features
    and ('medadmin_ingredient' not in x
         or x.endswith(('_has_medadmin_ingredient', 'earliest_minute')))
] + or_timings_features
print(f'Raw engineered predictors: {sum(col not in excluded for col in df_pivoted.columns)}')
print(f'Predictors entering SHAP ranking: {len(all_features)}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

from src.clinical_imputer import ClinicalImputer
from src.feature_selectors import CachedTopKSelector
from src.model_training import nested_cv_evaluation_feature_selection, tune_model_with_cv

selector_basemodel = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, min_samples_split=100,
)
pipeline = Pipeline([
    ('clinical', ClinicalImputer(columns=all_features)),
    ('selector', CachedTopKSelector(selector_basemodel, method='shap')),
    ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
])

k_grid = [k for k in [50, 100, 200, 300, 500, 800] if k <= len(all_features)]
param_grid = {
    'selector__k': k_grid or [len(all_features)],
    'clf__min_samples_split': [20, 50, 70, 100, 200, 500],
}
K_AUC_TOLERANCE = 0.005  # Absolute ROC-AUC units.
cv_settings = dict(
    groups=mother_patid, outer_splits=5, inner_splits=5, random_state=range(10),
    k_auc_tolerance=K_AUC_TOLERANCE,
)

# Each inner search chooses the smallest k within tolerance of its best mean AUC.
# Within that k, refit the highest-scoring RF setting. Outer-test data only score the refit.
# Enable only for analyses that need row-level predictions within the institution.
SAVE_INTERNAL_OOF = False
CachedTopKSelector.clean_cache()
try:
    metrics = nested_cv_evaluation_feature_selection(
        X, y, pipeline, param_grid, **cv_settings,
        return_num_features=True, return_inner_scores=True,
        return_for_curves=SAVE_INTERNAL_OOF, return_test_indices=SAVE_INTERNAL_OOF,
    )
finally:
    CachedTopKSelector.clean_cache()

In [ ]:
metrics

In [ ]:
metrics['number_of_features'].value_counts().sort_index()

In [ ]:
results_dir = Path('results/feature_selection_revision')
results_dir.mkdir(parents=True, exist_ok=True)
row_level = ['test_indices', 'y_true', 'y_probas']
metrics.drop(columns=['inner_scores', *row_level], errors='ignore').to_csv(
    results_dir / 'outer_fold_metrics.csv', index=False,
)
if SAVE_INTERNAL_OOF:
    # Restricted institutional output. Share only the aggregate files.
    internal_dir = results_dir / 'institution_only'
    internal_dir.mkdir(parents=True, exist_ok=True)
    predictions = pd.concat([
        pd.DataFrame({
            'repeat': row.repeat, 'outer_fold': row.outer_fold,
            'birthid': df_pivoted_preprocessed['birthid'].iloc[row.test_indices].to_numpy(),
            'mother_patid': mother_patid.iloc[row.test_indices].to_numpy(),
            'y_true': row.y_true, 'predicted_probability': row.y_probas,
        })
        for row in metrics.itertuples()
    ], ignore_index=True)
    predictions.to_parquet(internal_dir / 'outer_fold_predictions.parquet', index=False)
inner_scores = pd.DataFrame([
    {'repeat': row.repeat, 'outer_fold': row.outer_fold, **candidate}
    for row in metrics.itertuples() for candidate in row.inner_scores
])
inner_scores.to_csv(results_dir / 'inner_feature_search.csv', index=False)

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
best_by_k = inner_scores.groupby(['repeat', 'outer_fold', 'selector__k'])['mean_inner_score'].max()
auc_by_k = best_by_k.groupby('selector__k').mean().rename('mean_inner_auc')
auc_by_k.to_csv(results_dir / 'mean_inner_auc_by_k.csv')
auc_by_k.plot(marker='o', ax=axes[0])
axes[0].set(xlabel='Candidate feature count', ylabel='Mean inner-validation ROC-AUC')
metrics['number_of_features'].value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(xlabel='Selected feature count', ylabel='Outer folds',
            title=f'Smallest k within {K_AUC_TOLERANCE:g} AUC')
fig.tight_layout()
fig.savefig(results_dir / 'figure_3.png', bbox_inches='tight', dpi=200)

In [ ]:
selection_frequency = metrics['selected_features'].explode().value_counts().rename('outer_folds')
selection_frequency.to_csv(results_dir / 'feature_selection_frequency.csv')
selection_frequency.head(30)

In [ ]:
# Historical reduced-model ROC-AUC from submitted Table 2 (rounded).
summary = metrics[['roc_auc', 'ap', 'sens_80']].mean().to_frame().T
summary['submitted_roc_auc'] = 0.706
summary['roc_auc_change_from_submission'] = summary['roc_auc'] - summary['submitted_roc_auc']
summary.to_csv(results_dir / 'performance_summary.csv', index=False)
summary

In [ ]:
# Run the timing notebook first to also compare against the corrected full model.
full_metrics_path = Path('results/timing_revision/outer_fold_metrics.csv')
if full_metrics_path.is_file():
    import json

    config = json.loads(full_metrics_path.with_name('run_config.json').read_text())
    expected = {
        'rows': len(X), 'ssi_events': int(y.sum()), 'full_feature_names': all_features,
        'repeat_seeds': list(cv_settings['random_state']),
        'outer_splits': cv_settings['outer_splits'], 'inner_splits': cv_settings['inner_splits'],
        'rf_grid': {'clf__min_samples_split': param_grid['clf__min_samples_split']},
    }
    if any(config.get(key) != value for key, value in expected.items()):
        raise ValueError('Rerun the timing notebook with the same export and CV settings.')
    full = pd.read_csv(full_metrics_path)
    full = full[full['test'].eq('pcornet_has_med + med_earliest_minute + or_timings')]
    keys = ['repeat', 'outer_fold']
    paired = metrics[keys + ['roc_auc']].merge(
        full[keys + ['roc_auc']], on=keys, how='outer',
        suffixes=('_reduced', '_full'), validate='one_to_one',
    )
    if paired[['roc_auc_reduced', 'roc_auc_full']].isna().any().any():
        raise ValueError('Reduced and full runs must use the same repeat and fold settings.')
    paired['roc_auc_difference'] = paired['roc_auc_reduced'] - paired['roc_auc_full']
    paired.to_csv(results_dir / 'paired_roc_auc_differences.csv', index=False)
    print(paired['roc_auc_difference'].agg(['mean', 'std']))

In [ ]:
y.mean()

In [ ]:
metrics['best_params'].apply(lambda x: x['clf__min_samples_split']).describe()

In [ ]:
# One final development-cohort fit for the fixed predictor list and descriptive SHAP.
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
try:
    final_pipeline, clf = tune_model_with_cv(
        X, y, mother_patid, pipeline, param_grid, inner_cv,
        k_auc_tolerance=K_AUC_TOLERANCE,
    )
finally:
    CachedTopKSelector.clean_cache()
features_selected = final_pipeline[:-1].get_feature_names_out().tolist()
final_inner_scores = pd.DataFrame(clf.cv_results_['params'])
final_inner_scores['mean_inner_auc'] = clf.cv_results_['mean_test_score']
final_inner_scores.to_csv(results_dir / 'final_inner_feature_search.csv', index=False)

In [ ]:
clf.best_params_, len(features_selected)

In [ ]:
pd.DataFrame({'features': features_selected}).to_csv(
    results_dir / 'final_features_selected.csv', index=False
)

In [ ]:
import pickle
import json
from importlib.metadata import version

metadata = {
    'best_params': clf.best_params_,
    'n_estimators': final_pipeline.named_steps['clf'].n_estimators,
    'selected_features': features_selected,
    'candidate_k': param_grid['selector__k'],
    'k_auc_tolerance': K_AUC_TOLERANCE,
    'selected_mean_inner_auc': float(final_inner_scores.loc[clf.best_index_, 'mean_inner_auc']),
    'rf_grid': param_grid['clf__min_samples_split'],
    'repeat_seeds': list(cv_settings['random_state']),
    'outer_splits': cv_settings['outer_splits'], 'inner_splits': cv_settings['inner_splits'],
    'final_fit_seed': 42,
    'versions': {p: version(p) for p in ('numpy', 'pandas', 'scikit-learn', 'shap')},
}
with (results_dir / 'model_metadata.json').open('w') as fid:
    json.dump(metadata, fid, indent=2)
with (results_dir / 'final_pipeline.pickle').open('wb') as fid:
    pickle.dump(final_pipeline, fid)

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    explainer = shap.TreeExplainer(clf, data=X,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

X_selected = pd.DataFrame(
    final_pipeline[:-1].transform(X), columns=features_selected, index=X.index,
)
shap_values = get_shap(final_pipeline.named_steps['clf'], X_selected, max_display=50)

### 2. break down the shap

In [ ]:
import numpy as np

df_shap = pd.DataFrame({
    'feature': X_selected.columns,
    'shap': np.mean(np.abs(shap_values.values), axis=0)
}).sort_values(by='shap', ascending=False)

df_shap.shape

In [ ]:
df_shap.to_csv(results_dir / 'descriptive_final_model_shap.csv', index=False)

In [ ]:
df_shap

In [ ]:
feature_grouping = df.groupby(['description'])['feature_name'].unique().apply(list).to_dict()
list(feature_grouping.keys())

In [ ]:
import seaborn as sns
from matplotlib.ticker import MaxNLocator

todo = {
    'bp': ['all_bp', 'before_delivery_bp', 'after_delivery_bp'],
    'wt': ['wt', 'original_bmi', 'computed_bmi'],
    'lab': ['glucose', 'ebl'],
    'static': ['baseline'],
    'medication': ['before_delivery_medadmin_ingredient', 'after_delivery_medadmin_ingredient'],
    'diagnosis': ['diagnosis'],
    'OR timing': ['or_timings']
}

diagnosis_dict = {
    "O99_213_has_dx": "Obesity complicating pregnancy, third trimester",
    "O99_212_has_dx": "Obesity complicating pregnancy, second trimester",
    "E66_0_has_dx": "Obesity due to excess calories",
    "O09_892_has_dx": "Supervision of other high-risk pregnancies, second trimester",
    "O26_892_has_dx": "Other specified pregnancy-related conditions, second trimester",
    "O99_332_has_dx": "Smoking (tobacco) complicating pregnancy, second trimester",
    "Z13_79_has_dx": "Encounter for other screening for genetic and chromosomal anomalies",
    "O36_8130_has_dx": "Decreased fetal movements, third trimester, unsp",
    "Z98_891_has_dx": "History of uterine scar from previous surgery",
    "O99_824_has_dx": "Streptococcus B carrier state complicating childbirth"
}

med_dict = {
    '161': "Acetaminophen",
    '114202+1901+8591+9863': 'Lactated ringers',
    '1815': "Bupivacaine",
    '26225': "Ondansetron",
    '9863': "Sodium chloride (flush)",
    '6387': "Lidocaine",
    "2582": "Clindamycin",
    "5640": "Ibuprofen"
}

def replace_med_code(feature):
    for code, desc in med_dict.items():
        if feature.startswith(code):
            return feature.replace(code, desc, 1)
    return feature


def wrap_middle(text):
    if len(text) <= 30:
        return text
    parts = text.split()
    mid = len(parts) // 2
    return " ".join(parts[:mid]) + "\n" + " ".join(parts[mid:])


for group, key_list in todo.items():
    features = []
    for key in key_list:
        features.extend(feature_grouping.get(key, []))
    df_shap_filtered = df_shap[df_shap['feature'].isin(features)].copy()
    if df_shap_filtered.empty:
        continue

    # rename the column so more readable
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('has_medadmin_ingredient', 'Y/N')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('medadmin_ingredient', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('INTRAVENOUS', '(IV)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('ORAL', '(oral)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('UNKNOWN', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(replace_med_code)

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('diastolic_all', 'diastolic')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('systolic_all', 'systolic')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('bp_all', 'bp')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('or_timings', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('datetime', 'time')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('has_cmqcc_category', '_(cmqcc_category)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(lambda x: diagnosis_dict.get(x, x))

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('anes_', 'anesthesia_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('disch_', 'discharge_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('fac_', 'facility_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('sched_', 'scheduled_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('proc_', 'procedure_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('recov_', 'recovery_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('comp_', 'complete_')

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('_', ' ')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(wrap_middle)

    if group == 'diagnosis' or group == 'medication' or group == 'static' or group == 'OR timing':
        n = 10
        plt.figure(figsize=(3, 6))
    else:
        n = 5
        plt.figure(figsize=(3, 3))

    ax = sns.barplot(df_shap_filtered.head(n), x='shap', y='feature')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    plt.title(group)
    plt.ylabel('')

### 3. remake the shap plot using the same mapping

In [ ]:
df_shap_tmp = pd.DataFrame({'feature': shap_values.feature_names})

# rename the column so more readable
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('has_medadmin_ingredient', 'Y/N')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('medadmin_ingredient', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('INTRAVENOUS', '(IV)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('ORAL', '(oral)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('UNKNOWN', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].apply(replace_med_code)

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('diastolic_all', 'diastolic')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('systolic_all', 'systolic')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('bp_all', 'bp')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('or_timings', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('datetime', 'time')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('has_cmqcc_category', '_(cmqcc_category)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].apply(lambda x: diagnosis_dict.get(x, x))

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('anes_', 'anesthesia_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('disch_', 'discharge_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('fac_', 'facility_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('sched_', 'scheduled_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('proc_', 'procedure_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('recov_', 'recovery_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('comp_', 'complete_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('case_req', 'case_request')

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('_', ' ')

In [ ]:
from copy import deepcopy

shap_values_copy = deepcopy(shap_values)
# shap_values_copy.feature_names = df_shap_tmp['feature'].values.tolist()

top_idx = np.argsort(np.abs(shap_values_copy.values).mean(0))[-40:]

shap_values_copy.data = shap_values_copy.data[:, top_idx]
shap_values_copy.base_values = shap_values_copy.base_values[:]
shap_values_copy.values = shap_values_copy.values[:, top_idx]
shap_values_copy.feature_names = [df_shap_tmp['feature'].values[i] for i in top_idx]

ax = shap.plots.beeswarm(shap_values_copy, max_display=40, show=False)
ax.figure.savefig(results_dir / 'figure_4.png', bbox_inches='tight', dpi=200)
plt.show()

### 4 some more analysis for the best model

In [ ]:
X_selected.shape, len(features_selected), final_pipeline.named_steps['clf']

In [ ]:
metrics[['roc_auc', 'ap', 'sens_80', 'number_of_features']].describe()